# Part 2: NER Using Feed-Forward Neural Network

Uses the word embeddings learned in Part 1 (Skip-Gram with Negative Sampling).

Pipeline:
1. Load embeddings & vocab saved by Part 1
2. Build token-level dataset from CoNLL-2003 (train / val / test)
3. Train a Feed-Forward Neural Network (FFNN)
4. Evaluate: accuracy, precision, recall, F1-score
5. Plot: loss curves, confusion matrix, per-class F1

## 1. Imports

In [ ]:
import numpy as np
import re
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

## 2. Load Embeddings and Vocab from Part 1

In [ ]:
#Load embeddings & vocab from Part 1
embeddings = np.load('word_embeddings.npy')     # (VOCAB_SIZE, EMBED_DIM)
with open('vocab.pkl', 'rb') as f:
    vocab_data = pickle.load(f)
word2idx = vocab_data['word2idx']
idx2word = vocab_data['idx2word']
EMBED_DIM  = embeddings.shape[1]
 
print(f'Embeddings loaded: {embeddings.shape}')
print(f'Vocabulary size  : {len(word2idx)}')
print(f'Embedding dim     : {EMBED_DIM}')

Embeddings loaded: (13836, 100)
Vocabulary size  : 13836


## 3. Dataset Loading & Preprocessing

preprocess_token must be identical to the one used in Part 1 so that the vocabulary lookup is consistent.

In [ ]:
dataset = load_dataset('lhoestq/conll2003')

LABEL_LIST = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC", "B-MISC", "I-MISC"]
NUM_LABELS = len(LABEL_LIST)

print(f'Dataset splits: { {k: len(v) for k, v in dataset.items()} }')
print(f'NER labels ({NUM_LABELS}): {LABEL_LIST}')

In [ ]:
def preprocess_token(token: str) -> str:
    """
    Lowercase and remove non-alphanumeric characters except hyphens.
    Returns None for tokens that are purely punctuation/numbers.
    """
    token = token.lower()
    token = re.sub(r"[^a-z0-9\-]", "", token)
    return token if len(token) > 0 else None

In [50]:
def get_embedding(word):
    word = preprocess_token(word)
    if word and word in word2idx:
        return embeddings[word2idx[word]]
    return np.zeros(EMBED_DIM)

In [51]:
def build_split(split_name: str):

    #Returns:
        #X : list of np.ndarray, each shape (seq_len, EMBED_DIM)
        #y : list of np.ndarray, each shape (seq_len,)  int labels

    X_list = []
    y_list = []
    
    for example in dataset[split_name]:
        tokens = example['tokens']
        tags   = example['ner_tags']
        
        # Get embeddings for each token in the sentence
        for t, tag in zip(tokens, tags):
            vec = get_embedding(t)
            X_list.append(vec)
            y_list.append(tag)
            
    # simple list of vectors and a simple list of integers
    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.int64)
    return X, y

In [52]:
X_train, y_train = build_split('train')
X_val,   y_val   = build_split('validation')
X_test,  y_test  = build_split('test')
print(f'  Train : {len(X_train)} sentences')
print(f'  Val   : {len(X_val)} sentences')
print(f'  Test  : {len(X_test)} sentences')

  Train : 203621 sentences
  Val   : 51362 sentences
  Test  : 46435 sentences


## 4. Move Data to Device

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Using device: {device}')

X_train = torch.from_numpy(X_train).to(device)
y_train = torch.from_numpy(y_train).to(device)

X_val = torch.from_numpy(X_val).to(device)
y_val = torch.from_numpy(y_val).to(device)

X_test = torch.from_numpy(X_test).to(device)
y_test = torch.from_numpy(y_test).to(device)

## 5. Model Definition

Architecture: Embedding(100) → FC(512) → ReLU → Dropout(0.4) → FC(128) → ReLU → Dropout(0.4) → FC(9)

In [54]:
class NER_FFNN(nn.Module):
    def __init__(self, embedding_dim, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(embedding_dim, 512)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.4)
        self.fc2 = nn.Linear(512, 128)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.4)
        self.fc3 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.dropout1(self.relu1(self.fc1(x)))
        x = self.dropout2(self.relu2(self.fc2(x)))
        return self.fc3(x)

In [ ]:
model = NER_FFNN(embedding_dim=100, num_classes=NUM_LABELS).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 30
batch_size = 512

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {total_params:,}')
print(model)

## 6. Training Loop

In [ ]:
def train_epoch(X, y):
    model.train()
    total_loss = 0

    indices = torch.randperm(X.size(0))

    for i in range(0, X.size(0), batch_size):
        batch_idx = indices[i:i+batch_size]

        xb = X[batch_idx]
        yb = y[batch_idx]

        optimizer.zero_grad()

        outputs = model(xb)
        loss = criterion(outputs, yb)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / (X.size(0) // batch_size)


# for epoch in range(num_epochs):
#     train_loss = train_epoch(X_train, y_train)
#     val_loss = train_epoch(X_val, y_val)
    
#     print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

Epoch 1 | Train Loss: 0.5733 | Val Loss: 0.4376
Epoch 2 | Train Loss: 0.4282 | Val Loss: 0.4038
Epoch 3 | Train Loss: 0.4076 | Val Loss: 0.3861
Epoch 4 | Train Loss: 0.3926 | Val Loss: 0.3780
Epoch 5 | Train Loss: 0.3821 | Val Loss: 0.3678
Epoch 6 | Train Loss: 0.3745 | Val Loss: 0.3608
Epoch 7 | Train Loss: 0.3662 | Val Loss: 0.3525
Epoch 8 | Train Loss: 0.3598 | Val Loss: 0.3464
Epoch 9 | Train Loss: 0.3522 | Val Loss: 0.3399
Epoch 10 | Train Loss: 0.3446 | Val Loss: 0.3348
Epoch 11 | Train Loss: 0.3383 | Val Loss: 0.3283
Epoch 12 | Train Loss: 0.3340 | Val Loss: 0.3221
Epoch 13 | Train Loss: 0.3287 | Val Loss: 0.3195
Epoch 14 | Train Loss: 0.3231 | Val Loss: 0.3141
Epoch 15 | Train Loss: 0.3189 | Val Loss: 0.3142
Epoch 16 | Train Loss: 0.3149 | Val Loss: 0.3091
Epoch 17 | Train Loss: 0.3105 | Val Loss: 0.3066
Epoch 18 | Train Loss: 0.3066 | Val Loss: 0.3027
Epoch 19 | Train Loss: 0.3041 | Val Loss: 0.2984
Epoch 20 | Train Loss: 0.3008 | Val Loss: 0.2964
Epoch 21 | Train Loss: 0.2978

In [ ]:
def evaluate_loss(X, y):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for i in range(0, X.size(0), batch_size):
            xb = X[i:i+batch_size]
            yb = y[i:i+batch_size]
            outputs = model(xb)
            loss = criterion(outputs, yb)
            total_loss += loss.item()
    return total_loss / (X.size(0) // batch_size)

In [ ]:
train_losses, val_losses = [], []

for epoch in range(1, num_epochs + 1):
    train_loss = train_epoch(X_train, y_train)
    val_loss = evaluate_loss(X_val, y_val)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

## 7. Loss Curves

In [ ]:
epochs = range(1, num_epochs + 1)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(epochs, train_losses, marker='o', markersize=4, linewidth=2,
        color='steelblue', label='Train Loss')
ax.plot(epochs, val_losses,   marker='s', markersize=4, linewidth=2,
        color='coral',     label='Validation Loss')
ax.set_title('FFNN NER — Training & Validation Loss', fontsize=13, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Cross-Entropy Loss')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.35)
plt.tight_layout()
plt.savefig('ffnn_loss_curves.png', dpi=150)
plt.show()
print('Figure saved: ffnn_loss_curves.png')

## 8. Evaluation

In [ ]:
def get_predictions(X):
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, X.size(0), batch_size):
            out  = model(X[i:i + batch_size])
            preds.extend(torch.argmax(out, dim=1).cpu().numpy())
    return np.array(preds)


val_preds  = get_predictions(X_val)
test_preds = get_predictions(X_test)

y_val_np  = y_val.cpu().numpy()
y_test_np = y_test.cpu().numpy()

# Validation 
val_acc = accuracy_score(y_val_np, val_preds)
val_p, val_r, val_f1, _ = precision_recall_fscore_support(
    y_val_np, val_preds, average='weighted', zero_division=0
)
print(f'Validation - Accuracy: {val_acc:.4f}  F1: {val_f1:.4f}')

# Test
acc       = accuracy_score(y_test_np, test_preds)
p, r, f1, _ = precision_recall_fscore_support(y_test_np, test_preds,
                                               average='weighted', zero_division=0)
print('\n' + '=' * 45)
print('      FFNN - Test Set Results')
print('=' * 45)
print(f'  Accuracy  : {acc:.4f}')
print(f'  Precision : {p:.4f}')
print(f'  Recall    : {r:.4f}')
print(f'  F1-score  : {f1:.4f}')
print('=' * 45)


Validation F1: 0.9230

Test Results:
Accuracy : 0.9004
Precision: 0.8844
Recall   : 0.9004
F1-score : 0.8864


### 8.1 Per-Class Classification Report

In [ ]:
print(classification_report(
    y_test_np, test_preds,
    target_names=LABEL_LIST,
    zero_division=0
))

### 8.2 Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test_np, test_preds)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True, fmt='d',
    xticklabels=LABEL_LIST, yticklabels=LABEL_LIST,
    cmap='Blues', linewidths=0.5,
    ax=ax
)
ax.set_title('FFNN NER — Confusion Matrix (Test Set)', fontsize=13, fontweight='bold')
ax.set_xlabel('Predicted Tag')
ax.set_ylabel('True Tag')
plt.tight_layout()
plt.savefig('ffnn_confusion_matrix.png', dpi=150)
plt.show()
print('Figure saved: ffnn_confusion_matrix.png')

### 8.3 Per-Class F1 Score Bar Chart

In [ ]:
_, _, f1_per_class, support = precision_recall_fscore_support(
    y_test_np, test_preds,
    labels=list(range(NUM_LABELS)),
    zero_division=0
)

fig, ax = plt.subplots(figsize=(11, 5))
colors = ['#e74c3c' if s < 0.5 else '#f39c12' if s < 0.75 else '#2ecc71'
          for s in f1_per_class]
bars = ax.bar(LABEL_LIST, f1_per_class, color=colors, edgecolor='white', zorder=3)
ax.axhline(f1, color='steelblue', linestyle='--', linewidth=1.5,
           label=f'Weighted avg F1 = {f1:.4f}')
ax.set_ylim(0, 1.15)
ax.set_title('FFNN NER — Per-Class F1 Score (Test Set)', fontsize=13, fontweight='bold')
ax.set_xlabel('NER Tag')
ax.set_ylabel('F1 Score')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.35, zorder=0)

for bar, score, sup in zip(bars, f1_per_class, support):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.025,
            f'{score:.3f}\n(n={int(sup):,})', ha='center', va='bottom', fontsize=7.5)

plt.tight_layout()
plt.savefig('ffnn_f1_per_class.png', dpi=150)
plt.show()
print('Figure saved: ffnn_f1_per_class.png')

## 9. Summary

| Component | Details |
|---|---|
| Input | Word2Vec embeddings from Part 1 (dim = 100) |
| Architecture | FFNN: 100 → 512 → 128 → 9 |
| Activations | ReLU + Dropout(0.4) |
| Optimiser | Adam (lr = 0.001) |
| Loss | Cross-Entropy |
| Epochs | 30 |
| Batch size | 512 |
| Validation | Loss monitored each epoch (no weight updates on val) |
| Metrics | Accuracy, Precision, Recall, F1-score (weighted) |